# HW08-09: PyTorch MLP — регуляризация и оптимизация обучения

Часть S08: MLP, регуляризация (Dropout, BatchNorm, EarlyStopping)  
Часть S09: диагностика learning rate, Adam vs SGD+momentum, weight decay

## 2.3.1. Импорты, seed и устройство

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from pathlib import Path

# Фиксация seed для воспроизводимости
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda")
print(f"Device: {DEVICE}")

Device: cuda


## 2.3.2. Данные и DataLoader

In [6]:
# Датасет CIFAR10 (вариант C) — KMNIST недоступен, CIFAR10 скачивается с зеркал PyTorch
transform = transforms.Compose([transforms.ToTensor()])

train_full = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Разбиение train 80/20 на train и val с фиксированным seed
train_size = int(0.8 * len(train_full))
val_size = len(train_full) - train_size
train_dataset, val_dataset = random_split(train_full, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

100%|██████████| 170M/170M [00:20<00:00, 8.48MB/s] 


Train: 40000, Val: 10000, Test: 10000


## 2.3.3. Модель MLP и цикл обучения

In [8]:
# CIFAR10: 32x32x3 -> 3072 входа, 10 классов
INPUT_SIZE = 32 * 32 * 3
NUM_CLASSES = 10

class MLP(nn.Module):
    def __init__(self, hidden_sizes=(256, 128), dropout_p=0.0, use_batchnorm=False):
        super().__init__()
        self.flatten = nn.Flatten()
        layers = []
        in_dim = INPUT_SIZE
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            in_dim = h
        layers.append(nn.Linear(in_dim, NUM_CLASSES))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)

In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item()
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return total_loss / len(loader), correct / total

In [10]:
def run_training(model, train_loader, val_loader, criterion, optimizer, epochs, device,
                 early_stop_patience=None):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state = None
    no_improve = 0

    for ep in range(epochs):
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, va = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["train_acc"].append(ta)
        history["val_acc"].append(va)

        if va > best_val_acc:
            best_val_acc = va
            best_val_loss = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if (ep + 1) % 2 == 0 or ep == 0:
            print(f"Epoch {ep+1}/{epochs}  train_loss={tl:.4f} val_loss={vl:.4f} "
                  f"train_acc={ta:.4f} val_acc={va:.4f}")

        if early_stop_patience and no_improve >= early_stop_patience:
            print(f"Early stopping at epoch {ep+1}")
            break

    # Восстанавливаем лучшие веса при early stopping
    if best_state is not None and early_stop_patience:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return history, best_val_acc, best_val_loss, ep + 1

## 3. Часть A (S08): регуляризация (E1-E4)

In [11]:
# Папка для артефактов
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
(ARTIFACTS / "figures").mkdir(exist_ok=True)
runs_records = []

In [12]:
EPOCHS = 15
LR = 1e-3
criterion = nn.CrossEntropyLoss()

# E1 (base): MLP без Dropout и BatchNorm

In [13]:
model_e1 = MLP(hidden_sizes=(256, 128), dropout_p=0.0, use_batchnorm=False).to(DEVICE)
opt_e1 = optim.Adam(model_e1.parameters(), lr=LR)
hist_e1, best_val_e1, best_loss_e1, eps_e1 = run_training(
    model_e1, train_loader, val_loader, criterion, opt_e1, EPOCHS, DEVICE)
runs_records.append({
    "experiment_id": "E1", "dataset": "CIFAR10", "seed": SEED,
    "model_summary": "256-128, ReLU, no dropout, no BN",
    "optimizer": "Adam", "lr": LR, "momentum": "", "weight_decay": 0,
    "epochs_trained": eps_e1, "best_val_accuracy": best_val_e1, "best_val_loss": best_loss_e1
})

Epoch 1/15  train_loss=1.8946 val_loss=1.7595 train_acc=0.3135 val_acc=0.3706
Epoch 2/15  train_loss=1.6990 val_loss=1.6681 train_acc=0.3949 val_acc=0.4069
Epoch 4/15  train_loss=1.5780 val_loss=1.5682 train_acc=0.4354 val_acc=0.4426
Epoch 6/15  train_loss=1.4966 val_loss=1.5293 train_acc=0.4668 val_acc=0.4492
Epoch 8/15  train_loss=1.4437 val_loss=1.5182 train_acc=0.4829 val_acc=0.4590
Epoch 10/15  train_loss=1.3841 val_loss=1.5011 train_acc=0.5059 val_acc=0.4740
Epoch 12/15  train_loss=1.3511 val_loss=1.4306 train_acc=0.5180 val_acc=0.4965
Epoch 14/15  train_loss=1.3120 val_loss=1.4104 train_acc=0.5310 val_acc=0.4997


# E2 (Dropout): как E1 + Dropout(p=0.3)

In [14]:
model_e2 = MLP(hidden_sizes=(256, 128), dropout_p=0.3, use_batchnorm=False).to(DEVICE)
opt_e2 = optim.Adam(model_e2.parameters(), lr=LR)
hist_e2, best_val_e2, best_loss_e2, eps_e2 = run_training(
    model_e2, train_loader, val_loader, criterion, opt_e2, EPOCHS, DEVICE)
runs_records.append({
    "experiment_id": "E2", "dataset": "CIFAR10", "seed": SEED,
    "model_summary": "256-128, ReLU, Dropout(0.3)",
    "optimizer": "Adam", "lr": LR, "momentum": "", "weight_decay": 0,
    "epochs_trained": eps_e2, "best_val_accuracy": best_val_e2, "best_val_loss": best_loss_e2
})

Epoch 1/15  train_loss=2.0115 val_loss=1.8184 train_acc=0.2584 val_acc=0.3495
Epoch 2/15  train_loss=1.8587 val_loss=1.7715 train_acc=0.3293 val_acc=0.3649
Epoch 4/15  train_loss=1.7787 val_loss=1.6797 train_acc=0.3570 val_acc=0.4083
Epoch 6/15  train_loss=1.7410 val_loss=1.6603 train_acc=0.3740 val_acc=0.4168
Epoch 8/15  train_loss=1.7063 val_loss=1.6060 train_acc=0.3812 val_acc=0.4257
Epoch 10/15  train_loss=1.6745 val_loss=1.5747 train_acc=0.3978 val_acc=0.4411
Epoch 12/15  train_loss=1.6590 val_loss=1.5722 train_acc=0.4023 val_acc=0.4391
Epoch 14/15  train_loss=1.6422 val_loss=1.5506 train_acc=0.4066 val_acc=0.4493


# E3 (BatchNorm): как E1 + BatchNorm

In [15]:
model_e3 = MLP(hidden_sizes=(256, 128), dropout_p=0.0, use_batchnorm=True).to(DEVICE)
opt_e3 = optim.Adam(model_e3.parameters(), lr=LR)
hist_e3, best_val_e3, best_loss_e3, eps_e3 = run_training(
    model_e3, train_loader, val_loader, criterion, opt_e3, EPOCHS, DEVICE)
runs_records.append({
    "experiment_id": "E3", "dataset": "CIFAR10", "seed": SEED,
    "model_summary": "256-128, ReLU, BatchNorm",
    "optimizer": "Adam", "lr": LR, "momentum": "", "weight_decay": 0,
    "epochs_trained": eps_e3, "best_val_accuracy": best_val_e3, "best_val_loss": best_loss_e3
})

Epoch 1/15  train_loss=1.6441 val_loss=1.5633 train_acc=0.4134 val_acc=0.4461
Epoch 2/15  train_loss=1.4252 val_loss=1.5209 train_acc=0.4914 val_acc=0.4535
Epoch 4/15  train_loss=1.2301 val_loss=1.4867 train_acc=0.5597 val_acc=0.4753
Epoch 6/15  train_loss=1.0940 val_loss=1.6301 train_acc=0.6140 val_acc=0.4670
Epoch 8/15  train_loss=0.9835 val_loss=1.4593 train_acc=0.6523 val_acc=0.5021
Epoch 10/15  train_loss=0.8768 val_loss=1.7548 train_acc=0.6906 val_acc=0.4576
Epoch 12/15  train_loss=0.7773 val_loss=2.3103 train_acc=0.7268 val_acc=0.3894
Epoch 14/15  train_loss=0.6860 val_loss=1.7625 train_acc=0.7598 val_acc=0.4864


# E4 (EarlyStopping): выбираем лучший из E2/E3 по val_accuracy

In [ ]:
best_regularization = "E2" if best_val_e2 >= best_val_e3 else "E3"
use_dropout_e4 = best_regularization == "E2"
use_bn_e4 = best_regularization == "E3"

model_e4 = MLP(hidden_sizes=(256, 128), dropout_p=0.3 if use_dropout_e4 else 0.0,
               use_batchnorm=use_bn_e4).to(DEVICE)
opt_e4 = optim.Adam(model_e4.parameters(), lr=LR)
hist_e4, best_val_e4, best_loss_e4, eps_e4 = run_training(
    model_e4, train_loader, val_loader, criterion, opt_e4, EPOCHS, DEVICE,
    early_stop_patience=5)
runs_records.append({
    "experiment_id": "E4", "dataset": "CIFAR10", "seed": SEED,
    "model_summary": f"256-128, ReLU, {best_regularization}+EarlyStop(5)",
    "optimizer": "Adam", "lr": LR, "momentum": "", "weight_decay": 0,
    "epochs_trained": eps_e4, "best_val_accuracy": best_val_e4, "best_val_loss": best_loss_e4
})

Epoch 1/15  train_loss=1.6473 val_loss=1.5865 train_acc=0.4133 val_acc=0.4371
Epoch 2/15  train_loss=1.4159 val_loss=1.7785 train_acc=0.4949 val_acc=0.3898
Epoch 4/15  train_loss=1.2231 val_loss=1.5754 train_acc=0.5643 val_acc=0.4527
Epoch 6/15  train_loss=1.0914 val_loss=1.5447 train_acc=0.6116 val_acc=0.4712
Epoch 8/15  train_loss=0.9805 val_loss=1.4507 train_acc=0.6520 val_acc=0.5015
Epoch 10/15  train_loss=0.8669 val_loss=1.4610 train_acc=0.6942 val_acc=0.5169


## 3.2. Часть B (S09): LR, оптимизаторы, weight decay (O1-O3)

In [ ]:
# O1: LR слишком большой
model_o1 = MLP(hidden_sizes=(256, 128), dropout_p=0.3, use_batchnorm=False).to(DEVICE)
opt_o1 = optim.Adam(model_o1.parameters(), lr=1e-1)  # слишком большой lr
hist_o1, _, _, eps_o1 = run_training(model_o1, train_loader, val_loader, criterion, opt_o1, 6, DEVICE)
runs_records.append({
    "experiment_id": "O1", "dataset": "CIFAR10", "seed": SEED,
    "model_summary": "256-128, Dropout(0.3)", "optimizer": "Adam", "lr": 1e-1,
    "momentum": "", "weight_decay": 0, "epochs_trained": eps_o1,
    "best_val_accuracy": max(hist_o1["val_acc"]), "best_val_loss": min(hist_o1["val_loss"])
})

In [ ]:
# O2: LR слишком маленький
model_o2 = MLP(hidden_sizes=(256, 128), dropout_p=0.3, use_batchnorm=False).to(DEVICE)
opt_o2 = optim.Adam(model_o2.parameters(), lr=1e-5)  # слишком маленький lr
hist_o2, _, _, eps_o2 = run_training(model_o2, train_loader, val_loader, criterion, opt_o2, 6, DEVICE)
runs_records.append({
    "experiment_id": "O2", "dataset": "CIFAR10", "seed": SEED,
    "model_summary": "256-128, Dropout(0.3)", "optimizer": "Adam", "lr": 1e-5,
    "momentum": "", "weight_decay": 0, "epochs_trained": eps_o2,
    "best_val_accuracy": max(hist_o2["val_acc"]), "best_val_loss": min(hist_o2["val_loss"])
})

In [ ]:
# O3: SGD + momentum + weight decay
model_o3 = MLP(hidden_sizes=(256, 128), dropout_p=0.3, use_batchnorm=False).to(DEVICE)
opt_o3 = optim.SGD(model_o3.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)
hist_o3, best_val_o3, best_loss_o3, eps_o3 = run_training(
    model_o3, train_loader, val_loader, criterion, opt_o3, 12, DEVICE)
runs_records.append({
    "experiment_id": "O3", "dataset": "CIFAR10", "seed": SEED,
    "model_summary": "256-128, Dropout(0.3)", "optimizer": "SGD", "lr": 1e-2,
    "momentum": 0.9, "weight_decay": 1e-4, "epochs_trained": eps_o3,
    "best_val_accuracy": best_val_o3, "best_val_loss": best_loss_o3
})

## 4. Артефакты и финальная оценка

In [ ]:
# Сохранение runs.csv
import csv
with open(ARTIFACTS / "runs.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(runs_records[0].keys()))
    w.writeheader()
    w.writerows(runs_records)
print("Saved artifacts/runs.csv")

In [ ]:
# Сохранение лучшей модели (E4) и конфига
torch.save(model_e4.state_dict(), ARTIFACTS / "best_model.pt")
best_config = {
    "dataset": "CIFAR10", "seed": SEED,
    "hidden_sizes": [256, 128], "dropout_p": 0.3 if use_dropout_e4 else 0.0, "use_batchnorm": use_bn_e4,
    "optimizer": "Adam", "lr": LR, "epochs_trained": eps_e4,
    "best_val_accuracy": best_val_e4, "best_val_loss": best_loss_e4,
}
with open(ARTIFACTS / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, indent=2)
print("Saved best_model.pt and best_config.json")

In [ ]:
# График кривых для лучшего прогона (E4)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_e4["train_loss"], label="train")
ax1.plot(hist_e4["val_loss"], label="val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("E4: Loss")
ax1.legend()
ax2.plot(hist_e4["train_acc"], label="train")
ax2.plot(hist_e4["val_acc"], label="val")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("E4: Accuracy")
ax2.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS / "figures" / "curves_best.png", dpi=100)
plt.show()

In [ ]:
# График O1/O2 (LR слишком большой / маленький)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_o1["val_loss"], label="O1 val loss (lr=1e-1)")
ax1.plot(hist_o2["val_loss"], label="O2 val loss (lr=1e-5)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Val Loss")
ax1.set_title("LR extremes: Val Loss")
ax1.legend()
ax2.plot(hist_o1["val_acc"], label="O1 val acc (lr=1e-1)")
ax2.plot(hist_o2["val_acc"], label="O2 val acc (lr=1e-5)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Val Accuracy")
ax2.set_title("LR extremes: Val Accuracy")
ax2.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS / "figures" / "curves_lr_extremes.png", dpi=100)
plt.show()

In [ ]:
# Финальная оценка лучшей модели (E4) на test — один раз
test_loss, test_acc = evaluate(model_e4, test_loader, criterion, DEVICE)
print(f"Final test accuracy (E4): {test_acc:.4f}")